# 06B - SIMCA Final Pareto Selection

This notebook performs final multi-model selection after pure-test evaluation.

The selection is Pareto-based by track, without arbitrary final thresholds or weighted scores:

- `object_matrix_2way`: Pareto on `fn_rate` and `fp_rate`;
- `pixel_matrix_2way`: Pareto on `fn_rate` and `fp_rate`;
- `object_matrix_3way`: Pareto on `target_miss_rate`, `non_target_false_accept_rate`, and `uncertain_rate`;
- `pixel_matrix_3way`: Pareto on `target_miss_rate`, `non_target_false_accept_rate`, and `uncertain_rate`.

The first filter removes models dominated by another model on these error dimensions. A second, separate Pareto filter compares the previous-notebook flags as binary objectives (flag absent is better than flag present). Both filters are pairwise and are applied independently within each of the four tracks.


## Inputs And Outputs

Required inputs:

- `results/05_simca_validation_robustness_<RESULTS_TAG>/track_scoring_flags.parquet`
- `results/06A_simca_pure_test_<RESULTS_TAG>/pure_test_candidate_panel.parquet`
- `results/06A_simca_pure_test_<RESULTS_TAG>/pure_test_metrics_long.parquet`
- `results/06A_simca_pure_test_<RESULTS_TAG>/pure_test_guardrails.parquet`

Optional input:

- `results/06A_simca_pure_test_<RESULTS_TAG>/pure_test_errors.parquet`

Outputs:

- `final_selection_pool.parquet`: compact Pareto pool with selection status.
- `final_selected_models.parquet`: final selected multi-model set for notebook 07.
- `final_selection_summary.parquet`: compact status summary by track and Pareto tier.
- `final_selection_guardrails.parquet`: input guardrails for 06B.
- `final_selection_protocol.parquet`: settings and output counts.

The final-selection tables are intentionally compact. Full model configurations can be recovered from `pure_test_candidate_panel.parquet` by joining on `selected_config_id`.


In [1]:
from pathlib import Path
import sys
import pandas as pd

from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 250)

from src import experiment_config as expcfg
from src.utils import list_result_files
from src.workflows.simca_final_selection import (
    build_final_selection_guardrails,
    build_final_selection_pool,
    build_final_selection_protocol,
    save_final_selection_outputs,
    select_final_models_by_track,
    validate_final_selection_guardrails,
)
from src.workflows.simca_tables import read_simca_table, write_simca_table


PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


## Configuration

The first stage applies optional strict thresholds: `fn_rate < FN_RATE_MAX`, `fp_rate < FP_RATE_MAX`, and for 3way tracks `uncertain_rate < UNCERTAIN_RATE_MAX`. A `None` threshold skips that dimension. Pareto is applied only after this stage.

`TOP_N_FINAL_PER_TRACK` is optional. Set it to `None` to keep and display every model surviving both Pareto filters; set an integer only when an explicit output cap is wanted.

`APPLY_PREVIOUS_FLAG_FILTER` is optional and off by default. When enabled, it removes candidates carrying one of `PREVIOUS_FLAGS_TO_FILTER` before Pareto selection.

When diversity is enabled together with an integer cap, it only controls which survivors are shown under that cap; it never changes Pareto dominance.


In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_TAG = expcfg.DEFAULT_RESULTS_TAG

RESULTS_05_DIR = PROJECT_ROOT / "results" / f"05_simca_validation_robustness_{RESULTS_TAG}"
RESULTS_06A_DIR = PROJECT_ROOT / "results" / f"06A_simca_pure_test_{RESULTS_TAG}"
RESULTS_DIR = PROJECT_ROOT / "results" / f"06B_simca_final_selection_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRACK_SCORING_FLAGS_05_PATH = RESULTS_05_DIR / "track_scoring_flags.parquet"
PURE_TEST_CANDIDATE_PANEL_PATH = RESULTS_06A_DIR / "pure_test_candidate_panel.parquet"
PURE_TEST_METRICS_LONG_PATH = RESULTS_06A_DIR / "pure_test_metrics_long.parquet"
PURE_TEST_GUARDRAILS_PATH = RESULTS_06A_DIR / "pure_test_guardrails.parquet"
PURE_TEST_ERRORS_PATH = RESULTS_06A_DIR / "pure_test_errors.parquet"

FINAL_SELECTION_PATHS = {
    "pool": RESULTS_DIR / "final_selection_pool.parquet",
    "selected": RESULTS_DIR / "final_selected_models.parquet",
    "summary": RESULTS_DIR / "final_selection_summary.parquet",
    "guardrails": RESULTS_DIR / "final_selection_guardrails.parquet",
    "protocol": RESULTS_DIR / "final_selection_protocol.parquet",
}

TOP_N_FINAL_PER_TRACK = expcfg.SIMCA_FINAL_TOP_N_PER_TRACK
FN_RATE_MAX = expcfg.SIMCA_FINAL_FN_RATE_MAX
FP_RATE_MAX = expcfg.SIMCA_FINAL_FP_RATE_MAX
UNCERTAIN_RATE_MAX = expcfg.SIMCA_FINAL_UNCERTAIN_RATE_MAX
APPLY_DIVERSITY = expcfg.SIMCA_FINAL_APPLY_DIVERSITY
DIVERSITY_COLUMNS = tuple(expcfg.SIMCA_FINAL_DIVERSITY_COLUMNS)
DEDUPLICATE_ACROSS_TRACKS = expcfg.SIMCA_FINAL_DEDUPLICATE_ACROSS_TRACKS
CROSS_TRACK_DEDUP_COL = expcfg.SIMCA_FINAL_CROSS_TRACK_DEDUP_COL
TRACK_ORDER = tuple(expcfg.SIMCA_SELECTION_TRACKS)
REQUIRE_ALL_TRACKS = True

APPLY_PREVIOUS_FLAG_FILTER = expcfg.SIMCA_FINAL_APPLY_PREVIOUS_FLAG_FILTER
PREVIOUS_FLAGS_TO_FILTER = tuple(expcfg.SIMCA_FINAL_PREVIOUS_FLAGS_TO_FILTER)
EXCLUDE_PURE_TEST_ERRORS = expcfg.SIMCA_FINAL_EXCLUDE_PURE_TEST_ERRORS

print("Input 05 dir:", RESULTS_05_DIR)
print("Input 06A dir:", RESULTS_06A_DIR)
print("Output dir:", RESULTS_DIR)
print("TOP_N_FINAL_PER_TRACK (None = all survivors):", TOP_N_FINAL_PER_TRACK)
print("FN_RATE_MAX (strict):", FN_RATE_MAX)
print("FP_RATE_MAX (strict):", FP_RATE_MAX)
print("UNCERTAIN_RATE_MAX (strict, 3way only):", UNCERTAIN_RATE_MAX)
print("APPLY_DIVERSITY:", APPLY_DIVERSITY)
print("DIVERSITY_COLUMNS:", DIVERSITY_COLUMNS)
print("DEDUPLICATE_ACROSS_TRACKS:", DEDUPLICATE_ACROSS_TRACKS)
print("APPLY_PREVIOUS_FLAG_FILTER:", APPLY_PREVIOUS_FLAG_FILTER)
print("PREVIOUS_FLAGS_TO_FILTER:", PREVIOUS_FLAGS_TO_FILTER)


Input 05 dir: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\05_simca_validation_robustness_non_noisy_all
Input 06A dir: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06A_simca_pure_test_non_noisy_all
Output dir: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06B_simca_final_selection_non_noisy_all
TOP_N_FINAL_PER_TRACK (None = all survivors): None
FN_RATE_MAX (strict): 0.5
FP_RATE_MAX (strict): 0.9
UNCERTAIN_RATE_MAX (strict, 3way only): 0.6
APPLY_DIVERSITY: False
DIVERSITY_COLUMNS: ('preprocessing', 'rule_for_refit', 'balanced_pixel_strategy_effective')
DEDUPLICATE_ACROSS_TRACKS: False
APPLY_PREVIOUS_FLAG_FILTER: False
PREVIOUS_FLAGS_TO_FILTER: ()


## Load Inputs And Validate Guardrails

Notebook 06B requires notebook 05 robustness review and notebook 06A pure-test outputs. Pure-test guardrails must have passed before final selection starts.


In [3]:
required_paths = [
    TRACK_SCORING_FLAGS_05_PATH,
    PURE_TEST_CANDIDATE_PANEL_PATH,
    PURE_TEST_METRICS_LONG_PATH,
    PURE_TEST_GUARDRAILS_PATH,
]
missing_paths = [path for path in required_paths if not Path(path).exists()]
if missing_paths:
    raise FileNotFoundError("Missing required input file(s): " + ", ".join(map(str, missing_paths)))

track_scoring_flags_05_df = read_simca_table(TRACK_SCORING_FLAGS_05_PATH, required=True)
pure_test_candidate_panel_df = read_simca_table(PURE_TEST_CANDIDATE_PANEL_PATH, required=True)
pure_test_metrics_long_df = read_simca_table(PURE_TEST_METRICS_LONG_PATH, required=True)
pure_test_guardrails_df = read_simca_table(PURE_TEST_GUARDRAILS_PATH, required=True)
pure_test_errors_df = read_simca_table(PURE_TEST_ERRORS_PATH) if PURE_TEST_ERRORS_PATH.exists() else None

final_selection_guardrails_df = validate_final_selection_guardrails(
    build_final_selection_guardrails(
        track_scoring_flags_df=track_scoring_flags_05_df,
        pure_test_metrics_df=pure_test_metrics_long_df,
        pure_test_guardrails_df=pure_test_guardrails_df,
        candidate_panel_df=pure_test_candidate_panel_df,
        expected_tracks=TRACK_ORDER,
    )
)
write_simca_table(final_selection_guardrails_df, FINAL_SELECTION_PATHS["guardrails"])

print("05 track scoring flags:", track_scoring_flags_05_df.shape)
print("06A candidate panel:", pure_test_candidate_panel_df.shape)
print("06A pure-test metrics:", pure_test_metrics_long_df.shape)
print("06A pure-test errors:", None if pure_test_errors_df is None else pure_test_errors_df.shape)
display(final_selection_guardrails_df)


05 track scoring flags: (3964, 89)
06A candidate panel: (1982, 64)
06A pure-test metrics: (5946, 85)
06A pure-test errors: (0, 40)


,check_name,passed,status,severity,details,n_records
0,pure_test_guardrails_available,True,passed,error,,8.0
1,pure_test_guardrails_all_passed,True,passed,error,[],NaN
2,validation_review_table_available,True,passed,error,,3964.0
3,validation_review_tracks_complete,True,passed,error,[],NaN
4,pure_test_metrics_available,True,passed,error,,5946.0
5,pure_test_metric_tracks_complete,True,passed,error,[],NaN
6,candidate_panel_available,True,passed,warning,,1982.0


## Build Pareto Pool

The pool keeps one primary metric row per track and candidate, then adds previous-notebook flags and candidate metadata. No score and no final threshold is computed here.


In [4]:
cols_3way = ['model_candidate_id', 'selection_track',  'm_effective',
       'balanced_pixel_strategy_effective', 'preprocessing',
       'rule_for_refit', 'n_components', 'alpha',
       'object_threshold', 'sg_window_length', 'sg_polyorder',
       'position_dilation_radius', 
       'balanced_accuracy', 'fn_rate',
       'fp_rate',  'n_target', 'n_non_target', 'n_uncertain', 'uncertain_rate', 'coverage_rate',
       'target_miss_rate', 
       'screening_sensitivity', 'target_auto_accept_rate',
       'target_uncertain_rate', 'non_target_false_accept_rate',
       'non_target_auto_reject_rate', 'non_target_uncertain_rate',
       'decided_tp', 'decided_fn', 'decided_fp', 'decided_tn',]
cols_2way = ['model_candidate_id', 'selection_track',  'm_effective',
       'balanced_pixel_strategy_effective', 'preprocessing',
       'rule_for_refit', 'n_components', 'alpha',
       'object_threshold', 'sg_window_length', 'sg_polyorder',
       'position_dilation_radius', 
       'balanced_accuracy', 'fn_rate',
       'fp_rate']
mask_fn_fp = (pure_test_metrics_long_df['fn_rate'] < 0.9) & (pure_test_metrics_long_df['fp_rate'] < 0.9)
pure_test_metrics_long_df_obj_2w = pure_test_metrics_long_df[(pure_test_metrics_long_df['selection_track'] == 'object_matrix_2way') & mask_fn_fp]
pure_test_metrics_long_df_obj_3w = pure_test_metrics_long_df[(pure_test_metrics_long_df['selection_track'] == 'object_matrix_3way') & mask_fn_fp]
pure_test_metrics_long_df_px_2w = pure_test_metrics_long_df[(pure_test_metrics_long_df['selection_track'] == 'pixel_matrix_2way') & mask_fn_fp]
pure_test_metrics_long_df_px_3w = pure_test_metrics_long_df[(pure_test_metrics_long_df['selection_track'] == 'pixel_matrix_3way') & mask_fn_fp]

In [5]:
pure_test_metrics_long_df_px_2w[cols_2way]

,model_candidate_id,selection_track,m_effective,balanced_pixel_strategy_effective,preprocessing,rule_for_refit,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,balanced_accuracy,fn_rate,fp_rate
883,simca_d8a230c11f230302,pixel_matrix_2way,20.0,random,sg_smooth,simple_chi2,7,0.01,0.75,15,2,4,0.850934,0.068966,0.229167
884,simca_c9858542c9265482,pixel_matrix_2way,80.0,center,absorbance_snv_sg_smooth,simple_chi2,6,0.01,0.75,15,2,2,0.677083,0.000000,0.645833
885,simca_d19dcda42a4255aa,pixel_matrix_2way,40.0,random,sg_smooth,simple_chi2,7,0.01,0.75,11,2,3,0.823276,0.103448,0.250000
886,simca_3d88d635e74e703d,pixel_matrix_2way,40.0,random,sg_smooth,simple_chi2,7,0.01,0.80,11,2,3,0.837285,0.137931,0.187500
887,simca_fce640e13f8e3b03,pixel_matrix_2way,20.0,random,sg_smooth,simple_chi2,7,0.01,0.75,11,2,4,0.792026,0.103448,0.312500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3959,simca_647024fbabf6e8f1,pixel_matrix_2way,20.0,center,snv,alternative_chi2_fixed2,11,0.01,0.75,11,2,5,0.534393,0.171618,0.759595
3960,simca_297e5bdccb9f0b00,pixel_matrix_2way,20.0,center,absorbance_snv_sg_smooth,data_driven_chi2,12,0.01,0.75,9,2,4,0.576353,0.175727,0.671568
3961,simca_b6be95bff66f993e,pixel_matrix_2way,20.0,center,snv,simple_chi2,4,0.01,0.75,11,2,2,0.547398,0.186473,0.718732
3962,simca_0d8ee6e9533f09e5,pixel_matrix_2way,20.0,center,snv,data_driven_chi2,8,0.01,0.75,11,2,4,0.558367,0.238622,0.644644


In [6]:
final_selection_pool_df = build_final_selection_pool(
    pure_test_metrics_df=pure_test_metrics_long_df,
    track_scoring_flags_df=track_scoring_flags_05_df,
    candidate_panel_df=pure_test_candidate_panel_df,
    pure_test_errors_df=pure_test_errors_df,
    apply_previous_flag_filter=APPLY_PREVIOUS_FLAG_FILTER,
    previous_flags_to_filter=PREVIOUS_FLAGS_TO_FILTER,
    exclude_pure_test_errors=EXCLUDE_PURE_TEST_ERRORS,
)

print("Final Pareto pool:", final_selection_pool_df.shape)
display(final_selection_pool_df.groupby(["selection_track", "preselection_status"], dropna=False).size().reset_index(name="n_rows"))
display(final_selection_pool_df.head())


Final Pareto pool: (3964, 32)


,selection_track,preselection_status,n_rows
0,object_matrix_2way,candidate,883
1,object_matrix_3way,candidate,883
2,pixel_matrix_2way,candidate,1099
3,pixel_matrix_3way,candidate,1099


,selected_config_id,candidate_id,selection_track,matrix_family,decision_mode,metric_level,matrix_method,preprocessing,rule_for_refit,n_components,alpha,object_threshold,balanced_pixel_strategy_effective,n,fn_rate,fp_rate,balanced_accuracy,target_miss_rate,non_target_false_accept_rate,uncertain_rate,coverage_rate,decided_balanced_accuracy,previous_flags,selection_status,is_final_selected,preselection_status,filter_reason,previous_flag_count,previous_flag_filter_applied,filtered_by_previous_flags,has_pure_test_error,validation_metric_level
0,04C_refit_000008,simca_1c2855ffea473c9e,object_matrix_2way,object_matrix,2way,object,object_median,absorbance_sg_d2,data_driven_emp_cv,3,0.01,0.75,random,77,0.103448,0.479167,0.708693,NaN,NaN,NaN,NaN,NaN,high_fn_rate;high_fp_rate;low_balanced_accuracy,not_selected,False,candidate,,3,False,False,False,object
1,04C_refit_000009,simca_9af4221efa024d1f,object_matrix_2way,object_matrix,2way,object,object_median,absorbance_sg_d2,data_driven_emp_cv,3,0.01,0.75,random,77,0.103448,0.479167,0.708693,NaN,NaN,NaN,NaN,NaN,high_fn_rate;high_fp_rate;low_balanced_accuracy,not_selected,False,candidate,,3,False,False,False,object
2,04C_refit_000002,simca_a207c12cce354da3,object_matrix_2way,object_matrix,2way,object,object_median,absorbance_sg_d2,data_driven_emp_cv,10,0.01,0.75,random,77,0.034483,0.812500,0.576509,NaN,NaN,NaN,NaN,NaN,high_fp_rate;low_balanced_accuracy,not_selected,False,candidate,,2,False,False,False,object
3,04C_refit_000003,simca_08a074054db71001,object_matrix_2way,object_matrix,2way,object,object_median,absorbance_sg_d2,simple_emp_cv,10,0.01,0.75,random,77,0.034483,0.812500,0.576509,NaN,NaN,NaN,NaN,NaN,high_fp_rate;low_balanced_accuracy,not_selected,False,candidate,,2,False,False,False,object
4,04C_refit_000000,simca_0d9ef35630459e28,object_matrix_2way,object_matrix,2way,object,object_median,absorbance_sg_d2,data_driven_emp_cv,3,0.01,0.80,random,77,0.206897,0.479167,0.656968,NaN,NaN,NaN,NaN,NaN,high_fp_rate;low_balanced_accuracy,not_selected,False,candidate,,2,False,False,False,object


## Apply The Two Pareto Filters By Track

First, models dominated on the pure-test error dimensions are removed. Then, among the remaining models, flags from the previous notebooks are compared pairwise as separate binary Pareto objectives. Only the second-stage Pareto front is selected. `TOP_N_FINAL_PER_TRACK=None` keeps the complete front; no lower Pareto tier is used to fill a quota.


In [7]:
final_selected_models_df, final_selection_pool_df, final_selection_summary_df = select_final_models_by_track(
    final_selection_pool_df,
    top_n_per_track=TOP_N_FINAL_PER_TRACK,
    fn_rate_max=FN_RATE_MAX,
    fp_rate_max=FP_RATE_MAX,
    uncertain_rate_max=UNCERTAIN_RATE_MAX,
    apply_diversity=APPLY_DIVERSITY,
    diversity_columns=DIVERSITY_COLUMNS,
    deduplicate_across_tracks=DEDUPLICATE_ACROSS_TRACKS,
    cross_track_dedup_col=CROSS_TRACK_DEDUP_COL,
    track_order=TRACK_ORDER,
    require_all_tracks=REQUIRE_ALL_TRACKS,
)

print("Final selected models (second Pareto front):", final_selected_models_df.shape)
display(final_selected_models_df.groupby("selection_track", dropna=False).size().reset_index(name="n_selected"))
display(final_selection_summary_df)
display_columns = [
            "selection_track",
            "final_rank_in_track",
            "selection_status",
            "filter_reason",
            "pareto_tier",
            "pareto_rank_in_track",
            "selected_config_id",
            "candidate_id",
            "preprocessing",
            "rule_for_refit",
            "balanced_pixel_strategy_effective",
            "fn_rate",
            "fp_rate",
            "target_miss_rate",
            "non_target_false_accept_rate",
            "uncertain_rate",
        ]
display(final_selected_models_df[[col for col in display_columns if col in final_selected_models_df.columns]])


Final selected models (second Pareto front): (33, 30)


,selection_track,n_selected
0,object_matrix_2way,2
1,object_matrix_3way,7
2,pixel_matrix_2way,3
3,pixel_matrix_3way,21


,selection_track,selection_status,preselection_status,pareto_tier,n_rows,n_selected
0,object_matrix_2way,flag_pareto_dominated,candidate,2,7,0
1,object_matrix_2way,flag_pareto_dominated,candidate,3,6,0
2,object_matrix_2way,metric_pareto_dominated,candidate,<NA>,380,0
3,object_matrix_2way,rate_threshold_filtered,candidate,<NA>,488,0
4,object_matrix_2way,selected,candidate,1,2,2
5,object_matrix_3way,flag_pareto_dominated,candidate,2,5,0
6,object_matrix_3way,flag_pareto_dominated,candidate,3,9,0
7,object_matrix_3way,metric_pareto_dominated,candidate,<NA>,237,0
8,object_matrix_3way,rate_threshold_filtered,candidate,<NA>,625,0
9,object_matrix_3way,selected,candidate,1,7,7


,selection_track,final_rank_in_track,selection_status,pareto_tier,pareto_rank_in_track,selected_config_id,candidate_id,preprocessing,rule_for_refit,balanced_pixel_strategy_effective,fn_rate,fp_rate,target_miss_rate,non_target_false_accept_rate,uncertain_rate
0,object_matrix_2way,1,selected,1,1,04C_refit_000306,simca_2f532685804685f4,absorbance_sg_d1,data_driven_emp_cv,random,0.137931,0.333333,NaN,NaN,NaN
1,object_matrix_2way,2,selected,1,2,04C_refit_000290,simca_e6cd2d88651f1e6a,absorbance_sg_d1,data_driven_emp_cv,random,0.310345,0.229167,NaN,NaN,NaN
2,object_matrix_3way,1,selected,1,1,04C_refit_000430,simca_b237a6348e53cfc9,absorbance_sg_d2,simple_emp_cv,random,0.000000,0.729167,0.000000,0.729167,0.129870
3,object_matrix_3way,2,selected,1,2,04C_refit_000461,simca_c834a63e753e639e,absorbance_sg_d2,data_driven_emp_cv,random,0.000000,0.666667,0.000000,0.666667,0.168831
4,object_matrix_3way,3,selected,1,3,04C_refit_000597,simca_0e0daa82bb4f829f,absorbance_sg_d1,combined_index_chi2,random,0.000000,0.187500,0.000000,0.187500,0.389610
5,object_matrix_3way,4,selected,1,4,04C_refit_000724,simca_e4bfdaef8c2d9be2,absorbance_sg_d1,alternative_chi2_fixed2,random,0.413793,0.000000,0.413793,0.000000,0.220779
6,object_matrix_3way,5,selected,1,5,04C_refit_000726,simca_82315303eaf1abe9,absorbance_sg_d1,data_driven_emp_cv,random,0.000000,0.000000,0.000000,0.000000,0.415584
7,object_matrix_3way,6,selected,1,6,04C_refit_000741,simca_92aa706dd2fdc5b7,absorbance_sg_d1,alternative_chi2_fixed2,random,0.000000,0.083333,0.000000,0.083333,0.402597
8,object_matrix_3way,7,selected,1,7,04C_refit_000745,simca_0a287e88f7054bdc,absorbance_sg_d1,simple_emp_cv,random,0.241379,0.000000,0.241379,0.000000,0.272727
9,pixel_matrix_2way,1,selected,1,1,04C_refit_001484,simca_f2d13d914b5642c9,sg_smooth,alternative_chi2_fixed2,random,0.027813,0.595570,NaN,NaN,NaN


## Save Outputs

`final_selected_models.parquet` is the main output for notebook 07 mixture application.


In [8]:
protocol_settings = {
    "notebook": "06B_simca_final_selection",
    "results_tag": RESULTS_TAG,
    "input_05_dir": RESULTS_05_DIR,
    "input_06a_dir": RESULTS_06A_DIR,
    "top_n_final_per_track": (None if TOP_N_FINAL_PER_TRACK is None else int(TOP_N_FINAL_PER_TRACK)),
    "fn_rate_max": FN_RATE_MAX,
    "fp_rate_max": FP_RATE_MAX,
    "uncertain_rate_max": UNCERTAIN_RATE_MAX,
    "apply_diversity": bool(APPLY_DIVERSITY),
    "diversity_columns": DIVERSITY_COLUMNS,
    "deduplicate_across_tracks": bool(DEDUPLICATE_ACROSS_TRACKS),
    "cross_track_dedup_col": CROSS_TRACK_DEDUP_COL,
    "apply_previous_flag_filter": bool(APPLY_PREVIOUS_FLAG_FILTER),
    "previous_flags_to_filter": PREVIOUS_FLAGS_TO_FILTER,
    "exclude_pure_test_errors": bool(EXCLUDE_PURE_TEST_ERRORS),
}
final_selection_protocol_df = build_final_selection_protocol(
    protocol_settings,
    {
        "pool": final_selection_pool_df,
        "selected": final_selected_models_df,
        "summary": final_selection_summary_df,
    },
)

saved_paths = save_final_selection_outputs(
    pool_df=final_selection_pool_df,
    selected_df=final_selected_models_df,
    summary_df=final_selection_summary_df,
    guardrails_df=final_selection_guardrails_df,
    protocol_df=final_selection_protocol_df,
    paths=FINAL_SELECTION_PATHS,
)

print("Saved:")
for path in saved_paths:
    print(" -", path)

display(final_selection_protocol_df)
display(list_result_files(RESULTS_DIR))


Saved:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06B_simca_final_selection_non_noisy_all\final_selection_pool.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06B_simca_final_selection_non_noisy_all\final_selected_models.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06B_simca_final_selection_non_noisy_all\final_selection_summary.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06B_simca_final_selection_non_noisy_all\final_selection_guardrails.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\06B_simca_final_selection_non_noisy_all\final_selection_protocol.parquet


,notebook,results_tag,input_05_dir,input_06a_dir,apply_diversity,diversity_columns,deduplicate_across_tracks,cross_track_dedup_col,apply_previous_flag_filter,previous_flags_to_filter,exclude_pure_test_errors,n_pool_rows,n_candidate_rows,n_selected_rows,n_selected_tracks
0,06B_simca_final_selection,non_noisy_all,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,False,"[""preprocessing"", ""rule_for_refit"", ""balanced_...",False,selected_config_id,False,[],True,3964,3964,33,4


,file,suffixes,size_mb
0,final_selection_pool.parquet,.parquet,0.124163
1,final_selected_models.parquet,.parquet,0.021629
2,final_selection_protocol.parquet,.parquet,0.010441
3,final_selection_summary.parquet,.parquet,0.004197
4,final_selection_guardrails.parquet,.parquet,0.003768
